In [0]:
MY_ID = "2532Aastha"  
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.capstone_2532Aastha;
CREATE VOLUME IF NOT EXISTS workspace.capstone_2532Aastha.raw;

In [0]:
dbutils.fs.ls(f"/Volumes/workspace/capstone_{MY_ID}/raw")

[]

In [0]:
C = ['order_id','customer_id','order_ts','quantity','unit_price','channel','status']
CTY = "'Chennai','Pune','Kochi','Indore','Jaipur','Bhopal','Nagpur','Surat'"; R = f'{VOL}/raw'

def wr(df, path, cols):            # header on, whitespace preserved on write
    (df.selectExpr(*cols).write.mode('overwrite').option('header', True)
     .option('ignoreLeadingWhiteSpace', False)
     .option('ignoreTrailingWhiteSpace', False).csv(path))

def day(d, n, first, q="cast(1 + pmod(id, 5) as string)", c=None, t=None):
    return spark.range(n).selectExpr(
        f"format_string('ORD%06d', id + {first}) order_id",
        c or "format_string('C%05d', cast(pmod(id*7919, 3000) as int)) customer_id",
        t or f"cast(date_add(date'2025-06-01', {d-1}) as string) order_ts",
        f"{q} quantity",
        "round(20 + pmod(id*37, 18000)/100.0, 2) unit_price",
        "concat('CH', cast(pmod(id, 4) as string)) channel",
        "if(pmod(id, 50) = 0, 'CANCELLED', 'PLACED') status")

for d in range(1, 12):             # ten loaded files plus the clean control file
    wr(day(d, 5000, (d-1)*5000 + 1), f'{R}/orders/day_{d:02d}', C)

for d in range(1, 11):             # 300 customers a file, 3,000 in all
    wr(spark.range(300).selectExpr(
        f"format_string('C%05d', cast(id + {(d-1)*300} as int)) customer_id",
        f"element_at(array({CTY}), cast(pmod(id + {(d-1)*300}, 8) as int) + 1) city",
        f"cast(date_add(date'2025-06-01', {d-1}) as string) signup_date"),
        f'{R}/customers/cust_day_{d:02d}', ['customer_id','city','signup_date'])

wr(day(12, 5000, 55001, q="if(id < 40, 'unknown', cast(1+pmod(id,5) as string))"),
   f'{R}/orders/day_12', C)                          # 40 quantities not numeric

wr(day(13, 4850, 60001).union(day(1, 150, 1)), f'{R}/orders/day_13', C)

wr(day(14, 5000, 64851),                              # same names, moved order
   f'{R}/orders/day_14',
   ['order_id','order_ts','unit_price','quantity','status','channel','customer_id'])

spark.createDataFrame([(','.join(C),)], 'value string') \
    .write.mode('overwrite').text(f'{R}/orders/day_15')     # header line, no rows

wr(day(16, 5000, 69851,
       c="if(id < 220, 'C99999', format_string('C%05d', cast(pmod(id*7919, 3000) as int))) customer_id",
       t="if(id >= 220 and id < 310, '2035-06-16', '2025-06-16') order_ts"),
   f'{R}/orders/day_16', C)

In [0]:
for d in range(11, 17):
    p = f'{R}/orders/day_{d:02d}'
    n = spark.read.option('header', True).csv(p).count()
    print(d, n)

11 5000
12 5000
13 5000
14 5000
15 0
16 5000


In [0]:
from pyspark.sql import functions as F
import uuid, datetime

def bronze_load(path, table_name, load_id):
    raw_header_df = spark.read.option('header', True).csv(path)
    arrived_columns = raw_header_df.columns

    from pyspark.sql.types import StructType, StructField, StringType
    schema = StructType([StructField(c, StringType(), True) for c in arrived_columns])
    df = (spark.read.option('header', True).schema(schema).csv(path)
          .withColumn('_source_file', F.col('_metadata.file_path'))   # <-- changed line
          .withColumn('_ingested_at', F.current_timestamp())
          .withColumn('_load_id', F.lit(load_id))
          .withColumn('_row_hash', F.sha2(F.concat_ws('||', *arrived_columns), 256)))

    rows_in_file = df.count()

    manifest_row = spark.createDataFrame([(
        load_id, table_name, path, rows_in_file, ','.join(arrived_columns),
        datetime.datetime.utcnow()
    )], ['load_id', 'table_name', 'source_file', 'rows_in_file', 'column_list', 'ingested_at'])

    return df, manifest_row

In [0]:
# Orders: 10 loaded history files (day_01-day_10) + 6 new files (day_11-day_16)
order_files = [(f'{R}/orders/day_{d:02d}', 'orders') for d in range(1, 17)]

# Customers: 10 loaded history files (cust_day_01-cust_day_10)
customer_files = [(f'{R}/customers/cust_day_{d:02d}', 'customers') for d in range(1, 11)]

bronze_orders_dfs = []
bronze_customers_dfs = []
manifest_rows = []

for path, table_name in order_files + customer_files:
    load_id = str(uuid.uuid4())
    df, manifest_row = bronze_load(path, table_name, load_id)
    manifest_rows.append(manifest_row)
    if table_name == 'orders':
        bronze_orders_dfs.append(df)
    else:
        bronze_customers_dfs.append(df)

# Union each table's per-file DataFrames into one Bronze table
bronze_orders = bronze_orders_dfs[0]
for d in bronze_orders_dfs[1:]:
    bronze_orders = bronze_orders.unionByName(d, allowMissingColumns=True)

bronze_customers = bronze_customers_dfs[0]
for d in bronze_customers_dfs[1:]:
    bronze_customers = bronze_customers.unionByName(d)

bronze_load_manifest = manifest_rows[0]
for m in manifest_rows[1:]:
    bronze_load_manifest = bronze_load_manifest.unionByName(m)

# Write everything out, namespaced under your schema
bronze_orders.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.bronze_orders')
bronze_customers.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.bronze_customers')
bronze_load_manifest.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.bronze_load_manifest')

/home/spark-d56b5f02-921f-4455-8973-68/.ipykernel/71/command-5712244716146904-2217567361:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow()


In [0]:
print(spark.table(f'workspace.capstone_{MY_ID}.bronze_orders').count())      # expect 130,000 (50,000 history + 30,000... wait, count it yourself)
print(spark.table(f'workspace.capstone_{MY_ID}.bronze_customers').count())   # expect 3,000
print(spark.table(f'workspace.capstone_{MY_ID}.bronze_load_manifest').count())  # expect 26 (16 order files + 10 customer files)

75000
3000
26


In [0]:
silver_orders_raw = spark.table(f'workspace.capstone_{MY_ID}.bronze_orders')

silver_orders_typed = silver_orders_raw.selectExpr(
    'order_id',
    'customer_id',
    'try_cast(order_ts as date) as order_ts',
    'try_cast(quantity as int) as quantity',
    'try_cast(unit_price as double) as unit_price',
    'channel',
    'status',
    '_source_file', '_ingested_at', '_load_id', '_row_hash'
)

# R3: quantity_not_numeric — anything that failed try_cast is NULL
r3_rejects = silver_orders_typed.filter('quantity IS NULL')
print('R3 rejects (expect 40):', r3_rejects.count())

R3 rejects (expect 40): 40


In [0]:
# History = day_01 through day_10 (already "loaded" per the brief)
# New batch = day_11 through day_16 (what we're validating now)
history_pattern = r'day_(0[1-9]|10)/'
new_pattern = r'day_(1[1-6])/'

silver_orders_typed = silver_orders_typed.withColumn(
    '_day_num', F.regexp_extract('_source_file', r'day_(\d{2})', 1)
)

history_orders = silver_orders_typed.filter(F.col('_day_num').cast('int') <= 10)
new_orders     = silver_orders_typed.filter(F.col('_day_num').cast('int') >= 11)

print('history rows (expect 50000):', history_orders.count())
print('new rows before rejects (expect 25000):', new_orders.count())

history rows (expect 50000): 50000
new rows before rejects (expect 25000): 25000


In [0]:
# R1: null_key — order_id or customer_id missing (checklist says this should fire 0 times)
r1_rejects = new_orders.filter('order_id IS NULL OR customer_id IS NULL')
print('R1 rejects (expect 0):', r1_rejects.count())

# R2: duplicate_order_id — anti-join new batch against the TARGET TABLE as it already stands
# (i.e. history_orders), not against itself. This is the trap: dropDuplicates() on
# new_orders alone would report 0, because day_13's 150 repeats collide with day_01,
# not with each other.
r2_rejects = new_orders.join(
    history_orders.select('order_id'), on='order_id', how='left_semi'
)
print('R2 rejects (expect 150):', r2_rejects.count())

R1 rejects (expect 0): 0
R2 rejects (expect 150): 150


In [0]:
silver_customers = spark.table(f'workspace.capstone_{MY_ID}.bronze_customers').select('customer_id').distinct()

# R4: unknown_customer — customer_id has no match in the customer master
r4_rejects = new_orders.join(
    silver_customers, on='customer_id', how='left_anti'
)
print('R4 rejects (expect 220):', r4_rejects.count())

# R5: order_ts_out_of_range — range-check against real bounds, NOT the literal year 2035
# (the brief specifically warns against hardcoding 2035 — check the actual valid window instead)
r5_rejects = new_orders.filter(
    (F.col('order_ts') < F.lit('2024-01-01')) | (F.col('order_ts') > F.current_date())
)
print('R5 rejects (expect 90):', r5_rejects.count())

# Sanity check from your brief: R4 and R5 rejects should never overlap
overlap = r4_rejects.join(r5_rejects, on='order_id', how='inner').count()
print('R4/R5 overlap (expect 0):', overlap)

R4 rejects (expect 220): 220
R5 rejects (expect 90): 90
R4/R5 overlap (expect 0): 0


In [0]:
manifest = spark.table(f'workspace.capstone_{MY_ID}.bronze_load_manifest')

contract_orders_list = C  # ordered: ['order_id','customer_id','order_ts','quantity','unit_price','channel','status']
contract_customers_list = ['customer_id', 'city', 'signup_date']

def check_r6(row):
    arrived = row['column_list'].split(',')
    contract = contract_orders_list if row['table_name'] == 'orders' else contract_customers_list
    return arrived != contract   # ordered comparison, not set comparison

manifest_pd = manifest.select('load_id', 'table_name', 'source_file', 'column_list').toPandas()
manifest_pd['r6_fail'] = manifest_pd.apply(check_r6, axis=1)

r6_fails = manifest_pd[manifest_pd['r6_fail']]
print('R6 fails (expect 1, day_14):')
print(r6_fails[['table_name', 'source_file']])

R6 fails (expect 1, day_14):
   table_name                                        source_file
13     orders  /Volumes/workspace/capstone_2532Aastha/raw/raw...


In [0]:
manifest_full = manifest.select('load_id', 'table_name', 'source_file', 'rows_in_file', 'ingested_at').toPandas()
manifest_full = manifest_full.sort_values('ingested_at')

def check_r7(group):
    group = group.copy()
    fails = []
    for i in range(len(group)):
        window = group.iloc[max(0, i-7):i]  # last 7 loads BEFORE this one
        if len(window) == 0:
            fails.append(False)  # no history yet, can't judge
            continue
        median = window['rows_in_file'].median()
        n = group.iloc[i]['rows_in_file']
        fails.append(abs(n - median) > 0.4 * median if median > 0 else n > 0)
    group['r7_fail'] = fails
    return group

manifest_r7 = manifest_full.groupby('table_name', group_keys=False).apply(check_r7)
r7_fails = manifest_r7[manifest_r7['r7_fail']]
print('R7 fails (expect 1, day_15):')
print(r7_fails[['table_name', 'source_file', 'rows_in_file']])

R7 fails (expect 1, day_15):
   table_name                                        source_file  rows_in_file
14     orders  /Volumes/workspace/capstone_2532Aastha/raw/raw...             0


/home/spark-d56b5f02-921f-4455-8973-68/.ipykernel/71/command-5712244716146913-1543970582:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  manifest_r7 = manifest_full.groupby('table_name', group_keys=False).apply(check_r7)


In [0]:
# silver_orders = history (unchanged) + new rows that passed EVERY row-scope rule
r1_ids = r1_rejects.select('order_id')
r2_ids = r2_rejects.select('order_id')
r3_ids = r3_rejects.select('order_id')
r4_ids = r4_rejects.select('order_id')
r5_ids = r5_rejects.select('order_id')

all_reject_ids = r1_ids.union(r2_ids).union(r3_ids).union(r4_ids).union(r5_ids).distinct()

new_orders_accepted = new_orders.join(all_reject_ids, on='order_id', how='left_anti')

silver_orders = history_orders.drop('_day_num').unionByName(
    new_orders_accepted.drop('_day_num')
)

print('silver_orders count (expect 74500):', silver_orders.count())

# silver_rejects — one row per rejected order, tagged with which rule caught it
def tag(df, rule_id, rule_name):
    return df.select('order_id', '_load_id').withColumn('rule_id', F.lit(rule_id)) \
              .withColumn('rule_name', F.lit(rule_name))

silver_rejects = (
    tag(r1_rejects, 'R1', 'null_key')
    .unionByName(tag(r2_rejects, 'R2', 'duplicate_order_id'))
    .unionByName(tag(r3_rejects, 'R3', 'quantity_not_numeric'))
    .unionByName(tag(r4_rejects, 'R4', 'unknown_customer'))
    .unionByName(tag(r5_rejects, 'R5', 'order_ts_out_of_range'))
)

print('silver_rejects count (expect 500):', silver_rejects.count())

# Write both out
silver_orders.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.silver_orders')
silver_rejects.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.silver_rejects')

silver_orders count (expect 74500): 74500
silver_rejects count (expect 500): 500


In [0]:
# Flag every order row (history + new) against each row-scope rule
orders_flagged = (silver_orders_typed
    .withColumn('r1_fail', F.col('order_id').isNull() | F.col('customer_id').isNull())
    .withColumn('r3_fail', F.col('quantity').isNull())
    .withColumn('r5_fail', (F.col('order_ts') < F.lit('2024-01-01')) | (F.col('order_ts') > F.current_date()))
)

orders_flagged = orders_flagged.join(
    silver_customers.withColumn('_r4_match', F.lit(True)), on='customer_id', how='left'
).withColumn('r4_fail', F.col('_r4_match').isNull()).drop('_r4_match')

dup_ids = r2_rejects.select('order_id').distinct().withColumn('_is_dup', F.lit(True))
orders_flagged = orders_flagged.join(dup_ids, on='order_id', how='left') \
    .withColumn('r2_fail', F.col('_is_dup').isNotNull()).drop('_is_dup')

row_rule_meta = {'R1': ('r1_fail', 'null_key'), 'R2': ('r2_fail', 'duplicate_order_id'),
                  'R3': ('r3_fail', 'quantity_not_numeric'), 'R4': ('r4_fail', 'unknown_customer'),
                  'R5': ('r5_fail', 'order_ts_out_of_range')}

orders_row_rules = []
for rid, (col, rname) in row_rule_meta.items():
    df = (orders_flagged.groupBy('_load_id')
          .agg(F.count('*').alias('rows_checked'), F.sum(F.col(col).cast('int')).alias('rows_failed'))
          .withColumn('table_name', F.lit('orders'))
          .withColumn('rule_id', F.lit(rid))
          .withColumn('rule_name', F.lit(rname))
          .withColumn('rule_scope', F.lit('row')))
    orders_row_rules.append(df)

orders_row_df = orders_row_rules[0]
for d in orders_row_rules[1:]:
    orders_row_df = orders_row_df.unionByName(d)

# Customers only run R1
customers_flagged = spark.table(f'workspace.capstone_{MY_ID}.bronze_customers') \
    .withColumn('r1_fail', F.col('customer_id').isNull())

customers_row_df = (customers_flagged.groupBy('_load_id')
    .agg(F.count('*').alias('rows_checked'), F.sum(F.col('r1_fail').cast('int')).alias('rows_failed'))
    .withColumn('table_name', F.lit('customers'))
    .withColumn('rule_id', F.lit('R1'))
    .withColumn('rule_name', F.lit('null_key'))
    .withColumn('rule_scope', F.lit('row')))

print('orders row-rule rows (expect 80 = 16 loads x 5 rules):', orders_row_df.count())
print('customers row-rule rows (expect 10 = 10 loads x 1 rule):', customers_row_df.count())
print('SUM(rows_failed) across orders row rules (expect 500):', orders_row_df.agg(F.sum('rows_failed')).collect()[0][0])

orders row-rule rows (expect 80 = 16 loads x 5 rules): 75
customers row-rule rows (expect 10 = 10 loads x 1 rule): 10
SUM(rows_failed) across orders row rules (expect 500): 650


In [0]:
# Fix 1: join R2 dup flag on (order_id, _load_id) so only day_13's rows are flagged, not day_01's originals
dup_keys = r2_rejects.select('order_id', '_load_id').distinct().withColumn('_is_dup', F.lit(True))
orders_flagged = orders_flagged.drop('r2_fail').join(dup_keys, on=['order_id', '_load_id'], how='left') \
    .withColumn('r2_fail', F.col('_is_dup').isNotNull()).drop('_is_dup')

# Rebuild orders_row_rules with the corrected r2_fail
orders_row_rules = []
for rid, (col, rname) in row_rule_meta.items():
    df = (orders_flagged.groupBy('_load_id')
          .agg(F.count('*').alias('rows_checked'), F.sum(F.col(col).cast('int')).alias('rows_failed'))
          .withColumn('table_name', F.lit('orders'))
          .withColumn('rule_id', F.lit(rid))
          .withColumn('rule_name', F.lit(rname))
          .withColumn('rule_scope', F.lit('row')))
    orders_row_rules.append(df)

orders_row_df_raw = orders_row_rules[0]
for d in orders_row_rules[1:]:
    orders_row_df_raw = orders_row_df_raw.unionByName(d)

# Fix 2: fill in day_15 (and any other 0-row load) with an explicit 0/0 row per rule
all_order_load_ids = manifest.filter(F.col('table_name') == 'orders').select(
    F.col('load_id').alias('_load_id')
)
all_rule_ids = spark.createDataFrame([(r,) for r in row_rule_meta.keys()], ['rule_id'])
expected_grid = all_order_load_ids.crossJoin(all_rule_ids)

orders_row_df = (expected_grid.join(
        orders_row_df_raw, on=['_load_id', 'rule_id'], how='left')
    .withColumn('rows_checked', F.coalesce(F.col('rows_checked'), F.lit(0)))
    .withColumn('rows_failed', F.coalesce(F.col('rows_failed'), F.lit(0)))
    .withColumn('table_name', F.lit('orders'))
    .withColumn('rule_scope', F.lit('row'))
    .withColumn('rule_name', F.when(F.col('rule_name').isNotNull(), F.col('rule_name'))
                .otherwise(F.create_map(*[x for k, v in row_rule_meta.items() for x in (F.lit(k), F.lit(v[1]))])[F.col('rule_id')]))
)

print('orders row-rule rows (expect 80):', orders_row_df.count())
print('customers row-rule rows (expect 10):', customers_row_df.count())
print('SUM(rows_failed) across orders row rules (expect 500):', orders_row_df.agg(F.sum('rows_failed')).collect()[0][0])

orders row-rule rows (expect 80): 80
customers row-rule rows (expect 10): 10
SUM(rows_failed) across orders row rules (expect 500): 500


In [0]:
# Recompute full R6 flags (not just the failures) so we get pass rows too
manifest_pd_full = manifest.select('load_id', 'table_name', 'source_file', 'column_list').toPandas()
manifest_pd_full['r6_fail'] = manifest_pd_full.apply(check_r6, axis=1)

r6_spark = spark.createDataFrame(
    manifest_pd_full[['load_id', 'table_name', 'r6_fail']].rename(columns={'load_id': '_load_id'})
).withColumn('rule_id', F.lit('R6')).withColumn('rule_name', F.lit('column_contract')) \
 .withColumn('rule_scope', F.lit('file')).withColumn('rows_checked', F.lit(1)) \
 .withColumn('rows_failed', F.col('r6_fail').cast('int')).drop('r6_fail') \
 .filter(F.col('table_name') == 'orders')   # R6 only applies to orders per your checklist

r7_spark = spark.createDataFrame(
    manifest_r7[['load_id', 'table_name', 'r7_fail']].rename(columns={'load_id': '_load_id'})
).withColumn('rule_id', F.lit('R7')).withColumn('rule_name', F.lit('row_count_vs_median')) \
 .withColumn('rule_scope', F.lit('file')).withColumn('rows_checked', F.lit(1)) \
 .withColumn('rows_failed', F.col('r7_fail').cast('int')).drop('r7_fail')

print('R6 rows (expect 16, orders only):', r6_spark.count())
print('R7 rows (expect 26, orders + customers):', r7_spark.count())
print('R6 total failed (expect 1):', r6_spark.agg(F.sum('rows_failed')).collect()[0][0])
print('R7 total failed (expect 1):', r7_spark.agg(F.sum('rows_failed')).collect()[0][0])

R6 rows (expect 16, orders only): 16
R7 rows (expect 26, orders + customers): 26
R6 total failed (expect 1): 1
R7 total failed (expect 1): 1


In [0]:
common_cols = ['table_name', '_load_id', 'rule_id', 'rule_name', 'rule_scope', 'rows_checked', 'rows_failed']

customers_r7_only = r7_spark.filter(F.col('table_name') == 'customers')

gold_dq_results = (
    orders_row_df.select(*common_cols)
    .unionByName(customers_row_df.select(*common_cols))
    .unionByName(r6_spark.select(*common_cols))
    .unionByName(customers_r7_only.select(*common_cols))
    .unionByName(r7_spark.filter(F.col('table_name') == 'orders').select(*common_cols))
    .withColumnRenamed('_load_id', 'load_id')
    .withColumn('fail_rate', F.expr('try_divide(rows_failed, rows_checked)'))
    .withColumn('rule_status', F.when(F.col('rows_failed') > 0, 'fail').otherwise('pass'))
)

print('gold_dq_results total rows (expect 132):', gold_dq_results.count())
print('SUM(rows_failed) row-scope orders (expect 500):',
      gold_dq_results.filter((F.col('rule_scope')=='row') & (F.col('table_name')=='orders'))
      .agg(F.sum('rows_failed')).collect()[0][0])

gold_dq_results.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.gold_dq_results')

gold_dq_results total rows (expect 132): 132
SUM(rows_failed) row-scope orders (expect 500): 500


In [0]:
# "Now" = the latest legitimate business date across silver_orders (R5 already excluded the bad 2035 rows)
check_ts = silver_orders.agg(F.max('order_ts')).collect()[0][0]
print('check_ts (expect 2025-06-16):', check_ts)

# Latest business date per table
last_order_ts = check_ts  # by definition, orders' own latest date is check_ts
last_customer_ts = spark.table(f'workspace.capstone_{MY_ID}.bronze_customers') \
    .agg(F.max('signup_date')).collect()[0][0]
print('last_customer_ts (expect 2025-06-10):', last_customer_ts)

import datetime
check_ts_dt = datetime.date.fromisoformat(str(check_ts))
last_order_dt = datetime.date.fromisoformat(str(last_order_ts))
last_customer_dt = datetime.date.fromisoformat(str(last_customer_ts))

orders_hours_behind = (check_ts_dt - last_order_dt).days * 24
customers_hours_behind = (check_ts_dt - last_customer_dt).days * 24

gold_freshness = spark.createDataFrame([
    ('orders', str(check_ts_dt), str(last_order_dt), orders_hours_behind, 24,
     'pass' if orders_hours_behind <= 24 else 'fail'),
    ('customers', str(check_ts_dt), str(last_customer_dt), customers_hours_behind, 24,
     'pass' if customers_hours_behind <= 24 else 'fail'),
], ['table_name', 'check_ts', 'last_business_ts', 'hours_behind', 'expected_interval_hours', 'freshness_status'])

gold_freshness.show(truncate=False)
gold_freshness.write.mode('overwrite').saveAsTable(f'workspace.capstone_{MY_ID}.gold_freshness')

check_ts (expect 2025-06-16): 2025-06-16
last_customer_ts (expect 2025-06-10): 2025-06-10
+----------+----------+----------------+------------+-----------------------+----------------+
|table_name|check_ts  |last_business_ts|hours_behind|expected_interval_hours|freshness_status|
+----------+----------+----------------+------------+-----------------------+----------------+
|orders    |2025-06-16|2025-06-16      |0           |24                     |pass            |
|customers |2025-06-16|2025-06-10      |144         |24                     |fail            |
+----------+----------+----------------+------------+-----------------------+----------------+



In [0]:
export_path = f'{VOL}/export/gold_dq_results'

(gold_dq_results.coalesce(1)  # single CSV, "small enough to travel"
 .write.mode('overwrite').option('header', True).csv(export_path))

# Also export gold_freshness — your Gold layer has two tables, both need to travel
export_path_freshness = f'{VOL}/export/gold_freshness'
(gold_freshness.coalesce(1)
 .write.mode('overwrite').option('header', True).csv(export_path_freshness))

# Find the actual CSV filename Spark generated (it's not predictable — Spark names it itself)
dq_files = [f.name for f in dbutils.fs.ls(export_path) if f.name.endswith('.csv')]
fresh_files = [f.name for f in dbutils.fs.ls(export_path_freshness) if f.name.endswith('.csv')]
print('gold_dq_results CSV:', dq_files)
print('gold_freshness CSV:', fresh_files)

gold_dq_results CSV: ['part-00000-tid-6334074900918392722-6de8b69c-22dc-46d3-b5b9-08f6ed3a0624-1975-1-c000.csv']
gold_freshness CSV: ['part-00000-tid-4762549118628998941-c61c6147-5b53-4075-b872-3b87ddaf4f03-1976-1-c000.csv']
